# Run analysis and plot figures for manuscript

In [ ]:
### GENERAL SETUP
%matplotlib inline  
# this enables plotting within notebook

#import modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np   # basic math library  you will type np.$STUFF  e.g., np.cos(1)
import numpy.linalg as LA
from matplotlib.gridspec import GridSpec
import timeit
import cartopy.crs as ccrs
import datetime
import scipy.stats as stats # imports stats functions https://docs.scipy.org/doc/scipy/reference/stats.html
import cartopy.feature as cfeature
import cftime
import shapely
from cartopy.feature import ShapelyFeature
import matplotlib.patches as mpatches

In [ ]:
# Figure settings

import matplotlib as mpl
# Font style and size
# plt.rcParams['font.family'] = 'Arial'         # Font
plt.rcParams['font.size'] = 10                # General font size unless set below
plt.rcParams['axes.labelsize'] = 11           # Axes labels font size
plt.rcParams['figure.titlesize'] = 12         # Title font size
plt.rcParams['figure.titleweight'] = 'bold'   # Bold title
plt.rcParams['axes.labelweight'] = 'bold'     # Bold axes labels
    
# Axes and ticks parameters
plt.rcParams['axes.linewidth'] = 1            # Width of axes border
plt.rcParams['xtick.direction'] = 'in'        # Make x ticks go in
plt.rcParams['ytick.direction'] = 'in'        # Make y ticks go in
plt.rcParams['xtick.major.size'] = 5          # Set x tick length 
plt.rcParams['ytick.major.size'] = 5          # Set y tick length
plt.rcParams['xtick.major.width'] = 1         # Set x tick width 
plt.rcParams['ytick.major.width'] = 1         # Set y tick width

# Line style
plt.rcParams['lines.linewidth'] = 1           # Set line widths on plots
plt.rcParams['lines.linestyle'] = '-'         # Set line styles on plots

# Math text font characteristics
plt.rcParams['mathtext.fontset'] = 'cm'       # Choose font for math text
plt.rcParams['mathtext.default'] = 'regular'  # Make math text not bold or italic
# mathtext.FontConstantsBase.sup1 = 0.4         # Move superscript text to a better height

# For showing plots on GitHub
%matplotlib inline
plt.rcParams['figure.dpi']= 100

In [ ]:
def detrend_linear(dat, dim, order):
    """ linear detrend dat along the axis dim """
    params = dat.polyfit(dim=dim, deg=order)
    fit = xr.polyval(dat[dim], params.polyfit_coefficients)
    dat = dat-fit
    return dat, fit

## Load data and indices

In [ ]:
# open the data!
ds = xr.open_dataset(' /glade/work/smogen/CVP_bifurcation/data/SSH.regrid.nc')['SSH']
ds['time'] = pd.date_range("1958-01", "2020-12", freq="MS")
# center this bad boy on the Pacific
ds = ds.roll(lon=180,roll_coords=True)
ds['lon'] = np.arange(0.5,360.5,1)
# select the NE Pacific
ds = ds/100

In [ ]:
bifu_cesm2 = xr.open_dataset('CESM2.218.228.bifurcation.nc')
bifu_fosi = xr.open_dataset('FOSI.218.228.bifurcation.nc')
bifu_mer = xr.open_dataset('SODA.Lat.Bifurcation.nc')

core_fosi = xr.open_dataset('FOSI.Lat.Core.nc')['core']
core_cesm2 = xr.open_dataset('CESM2.Lat.Core.nc')['core']
core_mer = xr.open_dataset('SODA.Lat.Core.nc')['core']

str_fosi = xr.open_dataset('FOSI.UVEL.strength.nc')['strength']
str_cesm2 = xr.open_dataset('CESM2.UVEL.strength.nc')['strength']
str_mer = xr.open_dataset('MER.Core.Str.nc')['ugeo']

npgo_fosi = xr.open_dataset('FOSI.NPGO.nc')['PC2']
npgo_soda = xr.open_dataset('SODA.NPGO.nc')['PC2']
npgo_manu =  xr.open_dataset('MANU.NPGO.nc')['NPGO index']

pdo_fosi = -xr.open_dataset('FOSI.PDO.nc')['PC1']
pdo_soda = xr.open_dataset('SODA.PDO.nc')['PC1']
# pdo_obs = xr.open_dataset('pdo.timeseries.sstens.nc')['pdo']
# pdo_obs2 = xr.open_dataset('pdo.timeseries.ersstv5.nc')['pdo']

## cross correlations

In [ ]:
print('FOSI: core and bifu; r        =' + str(xr.corr(bifu_fosi,core_fosi,dim='time').values)) 
print('FOSI: core and bifu; r_12month=' + str(xr.corr(bifu_fosi.rolling(time=12).mean(),core_fosi.rolling(time=12).mean(),dim='time').values)) 

print('FOSI: core and str;  r        =' + str(xr.corr(str_fosi,core_fosi,dim='time').values)) 
print('FOSI: core and str;  r_12month=' + str(xr.corr(str_fosi.rolling(time=12).mean(),core_fosi.rolling(time=12).mean(),dim='time').values)) 

print('FOSI: bifu and str;  r        =' + str(xr.corr(str_fosi,bifu_fosi,dim='time').values)) 
print('FOSI: bifu and str;  r_12month=' + str(xr.corr(str_fosi.rolling(time=12).mean(),bifu_fosi.rolling(time=12).mean(),dim='time').values)) 

In [ ]:
from scipy.stats import linregress
m, b, r, p, err = linregress(bifu_fosi, core_fosi); print(p)
m, b, r, p, err = linregress(str_fosi, core_fosi); print(p)
m, b, r, p, err = linregress(bifu_fosi, str_fosi); print(p)

## Figure 4

In [ ]:
f, ax = plt.subplots(1,1,figsize=(10,5))

# plt.xlim('1958-01','2020-12')

t1,  = (-npgo_fosi).plot(color = 'k',linewidth=1,zorder=10, label = 'NPGO Index',alpha=0.5)
ax.set_ylim(-3,3)
ax.set_ylabel('NPGO Index')

ax2 = ax.twinx()
(core_fosi).plot(color='darkgoldenrod',linewidth=0.7,label='latitude of core',zorder=1,alpha=0.3)
t2,  = (core_fosi).rolling(time=12).mean().plot(color='darkgoldenrod',linewidth=2,label='latitude of core',zorder=1,alpha=1)
ax2.set_ylim(36,47)
ax2.set_ylabel('latitude')

ax3 = ax.twinx()
ax3.spines.right.set_position(("axes", 1.08))
(str_fosi/100).plot(color='rebeccapurple',linewidth=0.7,label='strength (m/s)',zorder=1, alpha=0.3)
t3,  = (str_fosi/100).rolling(time=12).mean().plot(color='rebeccapurple',linewidth=2,label='strength (m/s)',zorder=1, alpha=1)
ax3.set_ylim(0.02,0.12)

ax5 = ax.twinx()
ax5.spines.right.set_position(("axes", 1.18))
(bifu_fosi).plot(color = 'red',linewidth=0.7,label='latitude of bifurcation',zorder=1, alpha=0.3)
t4,  = (bifu_fosi).rolling(time=12).mean().plot(color = 'red',linewidth=2,label='latitude of bifurcation',zorder=1, alpha=1)
ax5.set_ylim(36,47)
# npgo_manu.plot(color = 'navy',linewidth=1.3)

fin_ax = ax.twinx()
fin_ax.spines.right.set_position(("axes", 1.3))
fin_ax.spines.right.set_visible(False)
t1,  = (-npgo_fosi).rolling(time=12).mean().plot(color = 'k',linewidth=2,zorder=10, label = 'NPGO Index')
fin_ax.set_ylim(-3,3); fin_ax.set_title('t')

ax.yaxis.label.set_color(t1.get_color())
ax2.yaxis.label.set_color(t2.get_color())
ax3.yaxis.label.set_color(t3.get_color())
ax5.yaxis.label.set_color(t4.get_color())

ax.tick_params(axis='y', colors=t1.get_color())
ax2.tick_params(axis='y', colors=t2.get_color())
ax3.tick_params(axis='y', colors=t3.get_color())
ax5.tick_params(axis='y', colors=t4.get_color())

# ax5.legend(handles=[t1, t2, t3, t4], loc = 'upper right', framealpha=1)

# plt.ylabel('strength (m/s)')
plt.xlabel('time')
plt.xlim('1958-01','2020-12')

plt.title('')

f.savefig('NPGO.TS.pdf',bbox_inches = "tight")

In [ ]:
print(xr.corr((-npgo_fosi).rolling(time=1).mean(), core_fosi.rolling(time=1).mean()).values)
print(xr.corr((-npgo_fosi).rolling(time=1).mean(), bifu_fosi.rolling(time=1).mean()).values)
print(xr.corr((-npgo_fosi).rolling(time=1).mean(), str_fosi.rolling(time=1).mean()).values)

print(xr.corr((-npgo_fosi).rolling(time=12).mean(), core_fosi.rolling(time=12).mean()).values)
print(xr.corr((-npgo_fosi).rolling(time=12).mean(), bifu_fosi.rolling(time=12).mean()).values)
print(xr.corr((-npgo_fosi).rolling(time=12).mean(), str_fosi.rolling(time=12).mean()).values)

In [ ]:
f, ax = plt.subplots(1,1,figsize=(10,5))


t1,  = (-pdo_fosi).plot(color = 'k',linewidth=1,zorder=10, label = 'NPGO Index',alpha=0.5)
ax.set_ylim(-3,3)
ax.set_ylabel('PDO Index')

ax2 = ax.twinx()
t2,  = (core_fosi).plot(color='darkgoldenrod',linewidth=0.7,label='latitude',zorder=1,alpha=0.3)
(core_fosi).rolling(time=12).mean().plot(color='darkgoldenrod',linewidth=2,label='latitude',zorder=1,alpha=1)
ax2.set_ylim(36,47)
ax2.set_ylabel('latitude')

ax3 = ax.twinx()
ax3.spines.right.set_position(("axes", 1.08))
t3,  = (str_fosi/100).plot(color='rebeccapurple',linewidth=0.7,label='strength (m/s)',zorder=1, alpha=0.3)
(str_fosi/100).rolling(time=12).mean().plot(color='rebeccapurple',linewidth=2,label='strength (m/s)',zorder=1, alpha=1)
ax3.set_ylim(0.02,0.12)

ax5 = ax.twinx()
ax5.spines.right.set_position(("axes", 1.18))
t4,  = (bifu_fosi).plot(color = 'red',linewidth=0.7,label='latitude ($^o$N))',zorder=1, alpha=0.3)
(bifu_fosi).rolling(time=12).mean().plot(color = 'red',linewidth=2,label='latitude ($^o$N))',zorder=1, alpha=1)
ax5.set_ylim(33,50)
# npgo_manu.plot(color = 'navy',linewidth=1.3)

fin_ax = ax.twinx()
fin_ax.spines.right.set_position(("axes", 1.3))
fin_ax.spines.right.set_visible(False)
t1,  = (-pdo_fosi).rolling(time=12).mean().plot(color = 'k',linewidth=2,zorder=10, label = 'PDO Index')
fin_ax.set_ylim(-3,3); fin_ax.set_title('t')

ax.yaxis.label.set_color(t1.get_color())
ax2.yaxis.label.set_color(t2.get_color())
ax3.yaxis.label.set_color(t3.get_color())
ax5.yaxis.label.set_color(t4.get_color())

ax.tick_params(axis='y', colors=t1.get_color())
ax2.tick_params(axis='y', colors=t2.get_color())
ax3.tick_params(axis='y', colors=t3.get_color())
ax5.tick_params(axis='y', colors=t4.get_color())

# ax4.legend(handles=[t1, t2, t3, t4], loc = 'upper right', framealpha=1)

# plt.ylabel('strength (m/s)')
plt.xlabel('time')
plt.xlim('1958-01','2020-12')

plt.title('')
f.savefig('PDO.TS.pdf',bbox_inches = "tight")

In [ ]:
print(xr.corr((pdo_fosi).rolling(time=1).mean(), core_fosi.rolling(time=1).mean()).values)
print(xr.corr((pdo_fosi).rolling(time=1).mean(), bifu_fosi.rolling(time=1).mean()).values)
print(xr.corr((pdo_fosi).rolling(time=1).mean(), str_fosi.rolling(time=1).mean()).values)

print(xr.corr((pdo_fosi).rolling(time=12).mean(), core_fosi.rolling(time=12).mean()).values)
print(xr.corr((pdo_fosi).rolling(time=12).mean(), bifu_fosi.rolling(time=12).mean()).values)
print(xr.corr((pdo_fosi).rolling(time=12).mean(), str_fosi.rolling(time=12).mean()).values)

## SF3

In [ ]:
f, ax = plt.subplots(1,1,figsize=(10,5))

# plt.xlim('1958-01','2020-12')

t1,  = (npgo_soda).plot(color = 'k',linewidth=1,zorder=10, label = 'NPGO Index',alpha=0.5)
ax.set_ylim(-3,3)
ax.set_ylabel('NPGO Index')

ax2 = ax.twinx()
(core_mer).plot(color='darkgoldenrod',linewidth=0.7,label='latitude of core',zorder=1,alpha=0.3)
t2,  = (core_mer).rolling(time=12).mean().plot(color='darkgoldenrod',linewidth=2,label='latitude of core',zorder=1,alpha=1)
ax2.set_ylim(36,47)
ax2.set_ylabel('latitude')

ax3 = ax.twinx()
ax3.spines.right.set_position(("axes", 1.08))
(str_mer).plot(color='rebeccapurple',linewidth=0.7,label='strength (m/s)',zorder=1, alpha=0.3)
t3,  = (str_mer).rolling(time=12).mean().plot(color='rebeccapurple',linewidth=2,label='strength (m/s)',zorder=1, alpha=1)
ax3.set_ylim(0.02,0.12)

ax5 = ax.twinx()
ax5.spines.right.set_position(("axes", 1.18))
(bifu_mer.lat).plot(color = 'red',linewidth=0.7,label='latitude of bifurcation',zorder=1, alpha=0.3)
t4,  = (bifu_mer.lat).rolling(time=12).mean().plot(color = 'red',linewidth=2,label='latitude of bifurcation',zorder=1, alpha=1)
ax5.set_ylim(36,47)
# npgo_manu.plot(color = 'navy',linewidth=1.3)

fin_ax = ax.twinx()
fin_ax.spines.right.set_position(("axes", 1.3))
fin_ax.spines.right.set_visible(False)
t1,  = (npgo_soda).rolling(time=12).mean().plot(color = 'k',linewidth=2,zorder=10, label = 'NPGO Index')
fin_ax.set_ylim(-3,3); fin_ax.set_title('t')

ax.yaxis.label.set_color(t1.get_color())
ax2.yaxis.label.set_color(t2.get_color())
ax3.yaxis.label.set_color(t3.get_color())
ax5.yaxis.label.set_color(t4.get_color())

ax.tick_params(axis='y', colors=t1.get_color())
ax2.tick_params(axis='y', colors=t2.get_color())
ax3.tick_params(axis='y', colors=t3.get_color())
ax5.tick_params(axis='y', colors=t4.get_color())

# ax5.legend(handles=[t1, t2, t3, t4], loc = 'upper right', framealpha=1)

# plt.ylabel('strength (m/s)')
plt.xlabel('time')
plt.xlim('1980-01','2020-12')

plt.title('')

f.savefig('NPGO.TS.SODA.pdf',bbox_inches = "tight")
# f.savefig('NPGO.TS.png',dpi=400,bbox_inches = "tight")

In [ ]:
f, ax = plt.subplots(1,1,figsize=(10,5))

t1,  = (pdo_soda).plot(color = 'k',linewidth=1,zorder=10, label = 'NPGO Index',alpha=0.5)
ax.set_ylim(-3,3)
ax.set_ylabel('PDO Index')

ax2 = ax.twinx()
t2,  = (core_mer).plot(color='darkgoldenrod',linewidth=0.7,label='latitude',zorder=1,alpha=0.3)
(core_mer).rolling(time=12).mean().plot(color='darkgoldenrod',linewidth=2,label='latitude',zorder=1,alpha=1)
ax2.set_ylim(36,47)
ax2.set_ylabel('latitude')

ax3 = ax.twinx()
ax3.spines.right.set_position(("axes", 1.08))
t3,  = (str_mer).plot(color='rebeccapurple',linewidth=0.7,label='strength (m/s)',zorder=1, alpha=0.3)
(str_mer).rolling(time=12).mean().plot(color='rebeccapurple',linewidth=2,label='strength (m/s)',zorder=1, alpha=1)
ax3.set_ylim(0.02,0.12)

ax5 = ax.twinx()
ax5.spines.right.set_position(("axes", 1.18))
t4,  = (bifu_mer.lat).plot(color = 'red',linewidth=0.7,label='latitude ($^o$N))',zorder=1, alpha=0.3)
(bifu_mer.lat).rolling(time=12).mean().plot(color = 'red',linewidth=2,label='latitude ($^o$N))',zorder=1, alpha=1)
ax5.set_ylim(33,50)
# npgo_manu.plot(color = 'navy',linewidth=1.3)

fin_ax = ax.twinx()
fin_ax.spines.right.set_position(("axes", 1.3))
fin_ax.spines.right.set_visible(False)
t1,  = (pdo_soda).rolling(time=12).mean().plot(color = 'k',linewidth=2,zorder=10, label = 'PDO Index')
fin_ax.set_ylim(-3,3); fin_ax.set_title('t')

ax.yaxis.label.set_color(t1.get_color())
ax2.yaxis.label.set_color(t2.get_color())
ax3.yaxis.label.set_color(t3.get_color())
ax5.yaxis.label.set_color(t4.get_color())

ax.tick_params(axis='y', colors=t1.get_color())
ax2.tick_params(axis='y', colors=t2.get_color())
ax3.tick_params(axis='y', colors=t3.get_color())
ax5.tick_params(axis='y', colors=t4.get_color())

# ax4.legend(handles=[t1, t2, t3, t4], loc = 'upper right', framealpha=1)

# plt.ylabel('strength (m/s)')
plt.xlabel('time')
plt.xlim('1980-01','2020-12')

plt.title('')
f.savefig('PDO.TS.SODA.pdf',bbox_inches = "tight")
# f.savefig('PDO.TS.png',dpi=400,bbox_inches = "tight")

## Figure 1

In [ ]:
ds_def = ds.sel(time='2001-01').squeeze('time')

In [ ]:
f, ax = plt.subplots(1,1, figsize=(14,5), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

im = ds_def.sel(lat = slice(20,62),lon = slice(178,245)).plot(cmap='RdBu_r',transform = ccrs.PlateCarree(),levels = np.arange(-0.5,0.51,0.01), extend = 'both', add_colorbar=False)

# (ds_def.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat')).plot(transform = ccrs.PlateCarree(),linewidth=1,color='k')


ax.add_feature(cfeature.LAND, color='k', zorder=3)
gl = ax.gridlines(crs=ccrs.PlateCarree(), linewidth=1, linestyle='--', color='black', alpha=0.3, draw_labels=True)
gl.top_labels = False
gl.right_labels = False

ax.add_patch(mpatches.Rectangle(xy=[180, 35], width=20, height=19,
                                    edgecolor='k',
                                    facecolor='none',
                                    transform=ccrs.PlateCarree(),linewidth=2)
                 )

ax.add_patch(mpatches.Rectangle(xy=[218, 37], width=10, height=19,
                                    edgecolor='k',
                                    facecolor='none',
                                    transform=ccrs.PlateCarree(),linewidth=2)
                 )

f.subplots_adjust(right=1.1)
cbar_ax = f.add_axes([0.85, 0.12, 0.015, 0.750])
cbar = f.colorbar(im, cax=cbar_ax, ticks=[-0.5,-0.4,-0.3,-0.2,-0.1,0,0.1,0.2,0.3,0.4,0.5])
# cbar = f.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=10)

# tmp = ShapelyFeature(lmes.geometry[9:10],ccrs.PlateCarree(), edgecolor='black')
# ax.add_feature(tmp, facecolor="None",lw=2)

# tmp = ShapelyFeature(lmes.geometry[3:4],ccrs.PlateCarree(), edgecolor='black')
# ax.add_feature(tmp, facecolor="None",lw=2)

f.savefig('ssh.map.with.box.pdf')

In [ ]:
f, ax = plt.subplots(1,1, figsize=(14,5), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

im = ds_def.differentiate('lat').sel(lat = slice(20,62),lon = slice(178,245)).plot(cmap='PRGn',transform = ccrs.PlateCarree(),levels = np.arange(-0.09,0.0905,0.0005), extend = 'both', add_colorbar=False)
ds_def.sel(lat = slice(37,55),lon = slice(178,260)).differentiate('lat').idxmin('lat').sel(lon = slice(180,200)).plot(transform = ccrs.PlateCarree(),color='k',linewidth=2)
# ds_def.sel(lat = slice(37,55),lon = slice(178,260)).differentiate('lat').idxmin('lat').sel(lon = slice(225,226)).plot(transform = ccrs.PlateCarree(),color='k',linewidth=2)

# plt.scatter(225.5, ds_def.sel(lat = slice(37,55),lon = slice(178,260)).differentiate('lat').idxmin('lat').sel(lon = slice(225,226)), transform = ccrs.PlateCarree(),marker='_',color='k')
ds_def.sel(lat = slice(37,55),lon = slice(178,260)).differentiate('lat').idxmin('lat').sel(lon = slice(218,228)).plot(transform = ccrs.PlateCarree(),color='k',linewidth=2)
# (ds_def.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat')).plot(transform = ccrs.PlateCarree(),linewidth=1,color='k')


ax.add_feature(cfeature.LAND, color='k', zorder=3)
gl = ax.gridlines(crs=ccrs.PlateCarree(), linewidth=1, linestyle='--', color='black', alpha=0.3, draw_labels=True)
gl.top_labels = False
gl.right_labels = False

ax.add_patch(mpatches.Rectangle(xy=[180, 35], width=20, height=19,
                                    edgecolor='k',
                                    facecolor='none',
                                    transform=ccrs.PlateCarree(),linewidth=2)
                 )

ax.add_patch(mpatches.Rectangle(xy=[218, 37], width=10, height=19,
                                    edgecolor='k',
                                    facecolor='none',
                                    transform=ccrs.PlateCarree(),linewidth=2)
                 )

f.subplots_adjust(right=1.1)
cbar_ax = f.add_axes([0.85, 0.12, 0.015, 0.750])
cbar = f.colorbar(im, cax=cbar_ax, ticks=[-0.08,0,0.08])
# cbar = f.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=10)

# tmp = ShapelyFeature(lmes.geometry[9:10],ccrs.PlateCarree(), edgecolor='black')
# ax.add_feature(tmp, facecolor="None",lw=2)

# tmp = ShapelyFeature(lmes.geometry[3:4],ccrs.PlateCarree(), edgecolor='black')
# ax.add_feature(tmp, facecolor="None",lw=2)

f.savefig('diff.map.with.box_line.pdf')

## Figure 5

In [ ]:
ds = xr.open_dataset('SSH.regrid.nc')['SSH']
ds['time'] = pd.date_range("1958-01", "2020-12", freq="MS")
# center this bad boy on the Pacific
ds = ds.roll(lon=180,roll_coords=True)
ds['lon'] = np.arange(0.5,360.5,1)
# select the NE Pacific
ds = ds/100

curl = xr.open_dataset('curl.regrid.nc')['curl']
curl['time'] = pd.date_range("1958-01", "2020-12", freq="MS")
curl = curl.roll(lon=180,roll_coords=True)
curl['lon'] = np.arange(0.5,360.5,1)

In [ ]:
f, ax = plt.subplots(1,1,figsize=(20/3,5),subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))


levels = [-0.75e-9, -0.5e-9, -0.25e-9, 0.25e-9, 0.5e-9, 0.75e-9]
# levels = [-1e-9, -0.5e-9, 0.5e-9,  1e-9]
# colors = ['k', 'k', 'k', 'k', 'w', 'w', 'w', 'w']
levels2 = [0]
colors2 = ['k']

(ds.mean('time')).plot(ax=ax,transform = ccrs.PlateCarree(), cmap='coolwarm',levels=np.arange(-0.5,0.52,0.02))
(curl.mean('time')).plot.contour(ax=ax,transform = ccrs.PlateCarree(), extend='both',linewidths=0.8, levels=levels,colors='k')
ts = (curl.mean('time')).plot.contour(ax=ax,transform = ccrs.PlateCarree(), extend='both',linewidths=1.5, levels=levels2,colors=colors2, linestyles='-.')

# ax.clabel(ts, fontsize=10)

# ax = ax.flatten()
# for i in range(len(ax)):
ax.add_feature(cfeature.LAND,color='k',zorder=10)
# ax.set_extent([-150,-120,30,55])
ax.set_extent([-180,-90,10,65])
ax.set_title('')

f.savefig('circulation.mean.FINAL.pdf')

In [ ]:
f, ax = plt.subplots(1,3,figsize=(20,5),subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

# set index
column = 2-2
index_to_use = core_fosi
index_name = 'core'
index_high, index_low = index_to_use.quantile(0.80), index_to_use.quantile(0.20)

# levels = [-0.5e-9, -0.3e-9, -0.1e-9, 0.1e-9, 0.3e-9, 0.5e-9]
# colors = ['k', 'k','k','w', 'w', 'w']
# levels = [-0.75e-9, -0.5e-9, -0.25e-9, 0.25e-9, 0.5e-9, 0.75e-9]
levels = [-0.25e-9, 0.25e-9]
# colors = ['k', 'k', 'dimgrey', 'w', 'w']

im = (ds.where(index_to_use > index_high).mean('time') - ds.where(index_to_use < index_low).mean('time')).plot(ax=ax[column],transform = ccrs.PlateCarree(), cmap='coolwarm',levels=np.arange(-0.1,0.11,0.01),add_colorbar=False)
tmp = (curl.where(index_to_use > index_high).mean('time') - curl.where(index_to_use < index_low).mean('time'))
ts = tmp.plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), colors='slategrey', levels=levels, extend='both')

levels2 = [0]
colors2 = ['slategrey']
tmp.plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), colors='slategrey', linewidths=1.3, linestyles='-.',levels=levels2)

levels2 = [0]
colors2 = ['k']
(curl.mean('time')).plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), extend='both',linewidths=1.3, levels=levels2,colors=colors2, linestyles='-.', alpha=1)

# set index
column = 3-2
index_to_use = str_fosi
index_name = 'strength'
index_high, index_low = index_to_use.quantile(0.80), index_to_use.quantile(0.20)

(ds.where(index_to_use > index_high).mean('time') - ds.where(index_to_use < index_low).mean('time')).plot(ax=ax[column],transform = ccrs.PlateCarree(), cmap='coolwarm',levels=np.arange(-0.1,0.11,0.01),add_colorbar=False)
tmp = (curl.where(index_to_use > index_high).mean('time') - curl.where(index_to_use < index_low).mean('time'))
ts = tmp.plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), colors='slategrey', levels=levels, extend='both')

levels2 = [0]
colors2 = ['slategrey']
tmp.plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), colors='slategrey', linewidths=1.3, linestyles='-.',levels=levels2)

levels2 = [0]
colors2 = ['k']
(curl.mean('time')).plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), extend='both',linewidths=1.3, levels=levels2,colors=colors2, linestyles='-.', alpha=1)

# set index
column = 4-2
index_to_use = bifu_fosi
index_name = 'bifurcation'
index_high, index_low = index_to_use.quantile(0.80), index_to_use.quantile(0.20)

im = (ds.where(index_to_use > index_high).mean('time') - ds.where(index_to_use < index_low).mean('time')).plot(ax=ax[column],transform = ccrs.PlateCarree(), cmap='coolwarm',levels=np.arange(-0.1,0.11,0.01),add_colorbar=False)
tmp = (curl.where(index_to_use > index_high).mean('time') - curl.where(index_to_use < index_low).mean('time'))
ts = tmp.plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), colors='slategrey', levels=levels, extend='both')

levels2 = [0]
colors2 = ['slategrey']
tmp.plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), colors='slategrey', linewidths=1.3, linestyles='-.',levels=levels2)

levels2 = [0]
colors2 = ['k']
(curl.mean('time')).plot.contour(ax=ax[column],transform = ccrs.PlateCarree(), extend='both',linewidths=1.3, levels=levels2,colors=colors2, linestyles='-.', alpha=1)

ax = ax.flatten()
for i in range(len(ax)):
    ax[i].add_feature(cfeature.LAND,color='k',zorder=10)
    # ax[i].set_extent([-150,-120,30,55])
    ax[i].set_extent([-180,-90,10,65])
    ax[i].set_title('')

f.subplots_adjust(right=0.83); cbar_ax = f.add_axes([0.85, 0.12, 0.015, 0.750]); cbar = f.colorbar(im, cax=cbar_ax); cbar.ax.tick_params(labelsize=10)

f.savefig('circulation.changes.3panel.FINAL.pdf')

## Figure 2

In [ ]:
# Plot the actual historical evolution of the indices
print(xr.corr((core_fosi).rolling(time=6,center=True).mean(), (core_mer).rolling(time=6,center=True).mean()))

# compare Latitude of Core
f, ax = plt.subplots(1,1,figsize=(10,3))
(core_fosi).plot(color='navy',linewidth=1,alpha=0.5)
(core_fosi).rolling(time=6,center=True).mean().plot(color='navy',linewidth=1.6,label='FOSI')

(core_cesm2.mean('member')).plot(color='lightblue',linewidth=1.5,label='CESM2-LE',alpha=0)
(core_cesm2).plot.line(hue='member',color='lightblue',linewidth=0.1,alpha=1,zorder=0)
(core_mer).plot(color='k',linewidth=1,alpha=0.5)
(core_mer).rolling(time=6,center=True).mean().plot(color='k',linewidth=1.6,label='SODA')

plt.xlim(core_fosi.time[0],core_fosi.time[-1])

# corr = str(np.round(xr.corr(core_fosi, core_mer).values,3))
# plt.text(core_fosi.time[8], 36, 'r = ' + corr,fontsize=12)
# corr_roll = str(np.round(xr.corr(core_fosi.rolling(time=12).mean(), core_mer.rolling(time=12).mean()).values,3))
# plt.text(core_fosi.time[8], 35.2, 'r$_{rolling, 12 month}$ = ' + corr_roll,fontsize=12)

plt.legend(loc = 'upper left',fontsize=12)

plt.ylabel('latitude')
plt.xlabel('time')

plt.ylim(36,47)
# plt.title('Lat. of Core Comparison')

f.savefig('core.historical.pdf')

In [ ]:
# compare Strength
print(xr.corr((str_mer).rolling(time=6,center=True).mean(), (str_fosi/100).rolling(time=6,center=True).mean()))
f, ax = plt.subplots(1,1,figsize=(10,3))
(str_fosi/100).plot(color='navy',linewidth=1,alpha=0.5)
(str_fosi/100).rolling(time=6,center=True).mean().plot(color='navy',linewidth=1.6,label='FOSI')
(str_cesm2.mean('member')/100).plot(color='lightblue',linewidth=1.6,label='CESM2-LE',alpha=0)
(str_cesm2/100).plot.line(hue='member',color='lightblue',linewidth=0.1,alpha=1,zorder=0,add_legend=False)
(str_mer).plot(color='k',linewidth=1,alpha=0.5,zorder=0)
(str_mer).rolling(time=6,center=True).mean().plot(color='k',linewidth=1.6,label='SODA',zorder=0)

plt.xlim(core_fosi.time[0],core_fosi.time[-1])
# plt.ylim(26,56)
# corr = str(np.round(xr.corr(str_fosi/100, str_mer).values,3))
# plt.text(core_fosi.time[8], 0.01, 'r = ' + corr,fontsize=12)
# corr_roll = str(np.round(xr.corr(str_fosi.rolling(time=12).mean()/100, str_mer.rolling(time=12).mean()).values,3))
# plt.text(core_fosi.time[8], -0.01, 'r$_{rolling, 12 month}$ = ' + corr_roll,fontsize=12)

# plt.legend(loc='lower right')

plt.ylabel('strength (m/s)')
plt.xlabel('time')

plt.ylim(0,0.18)
# plt.title('Strength')

f.savefig('strength.historical.pdf')

In [ ]:
# compare Strength
f, ax = plt.subplots(1,1,figsize=(10,3))

(bifu_fosi).plot(color='navy',linewidth=1,alpha=0.5)
(bifu_fosi).rolling(time=6,center=True).mean().plot(color='navy',linewidth=1.6,label='FOSI')

(bifu_cesm2.lat.mean('member')).plot(color='lightblue',linewidth=1.6,label='CESM2-LE',alpha=0)
(bifu_cesm2.lat).plot.line(hue='member',color='lightblue',linewidth=0.1,alpha=1,zorder=0,add_legend=False)

(bifu_mer).lat.plot(color='k',linewidth=1,alpha=0.5,zorder=0)
(bifu_mer).lat.rolling(time=6,center=True).mean().plot(color='k',linewidth=1.4,label='SODA',zorder=0)

plt.xlim(bifu_fosi.time[0],bifu_fosi.time[-1])
# plt.ylim(26,56)
# corr = str(np.round(xr.corr(str_fosi/100, str_mer).values,3))
# plt.text(core_fosi.time[8], 0.01, 'r = ' + corr,fontsize=12)
# corr_roll = str(np.round(xr.corr(str_fosi.rolling(time=12).mean()/100, str_mer.rolling(time=12).mean()).values,3))
# plt.text(core_fosi.time[8], -0.01, 'r$_{rolling, 12 month}$ = ' + corr_roll,fontsize=12)

# plt.legend(loc='lower right')

plt.ylabel('latitude ($^o$N)')
plt.xlabel('time')
# plt.legend()
plt.ylim(36,55)
# plt.title('Strength')

f.savefig('bifu.historical.pdf')

## Figure 3

In [ ]:
# dat, fit = detrend_linear(core_fosi.sel(time=slice('1970','2009')), 'time', 1)
dat, fit = detrend_linear(core_fosi.sel(time=slice('1970','2010')), 'time', 1)

dat2, fit2 = detrend_linear((core_cesm2).sel(time=slice(fit.time[0],fit.time[-1])), 'time', 1)
# subset for plotting purposes
fit2_subs  = fit2.isel(member=slice(30,40))

# mean slope across members
dat3, fit3 = detrend_linear((core_cesm2.mean('member')).sel(time=slice(fit.time[0],fit.time[-1])), 'time', 1)

core_cesm2_plot = core_cesm2.isel(member=slice(30,40))

In [ ]:
# slope of linear trend for all members
slopes = ((fit2[1] - fit2[0]) * 12 * 10)

In [ ]:
# Plot the actual historical evolution of the indices

# compare Latitude of Core
f, ax = plt.subplots(1,1,figsize=(8,5))
(core_fosi).plot(color='navy',linewidth=0.5,zorder=100)
(core_fosi).rolling(time=6).mean().plot(color='navy',linewidth=1,label='FOSI',zorder=100)

(core_cesm2.isel(member=11)).plot(color='lightblue',linewidth=1, alpha=0,label='CESM2-LE')
# core_cesm2.mean('member').plot(color='navy',linewidth=1, alpha=1,label='CESM2-LE')

(core_cesm2_plot).plot.line(hue='member',color='lightblue',linewidth=0.7,alpha=1,zorder=0)

plt.xlim(fit.time[0],fit.time[-1])
(fit).plot(color='k',linewidth=3,zorder=102, linestyle='--', label = 'FOSI Trend')

fit2_subs.isel(member=0).plot.line(color='dimgrey',linewidth=3,alpha=0.0,zorder=101,linestyle='--', label = 'CESM2-LE Trend')

fit3.plot.line(color='dimgrey',linewidth=3,alpha=0.9,zorder=101,linestyle='--')

plt.hist(slopes, density=True, bins='auto', histtype='stepfilled', alpha=0.5)

plt.axvline(((fit[1] - fit[0]) * 12 * 10))
plt.legend(loc = 'upper left',fontsize=12)

plt.ylabel('latitude')
plt.xlabel('time')

plt.ylim(36,47)

f.savefig('core.spread.trend.new.pdf')

In [ ]:
f, ax = plt.subplots(1,1,figsize=(2,5))

plt.hist(slopes, bins = [-0.6,-0.4,-0.2,0.0,0.2,0.4,0.6, 0.8,1], density=True, histtype='stepfilled', alpha=0.3, orientation='horizontal',color = 'dimgrey',align='left')
plt.axhline(((fit[1] - fit[0]) * 12 * 10),color='k', linewidth = 3,linestyle='--')
plt.axhline(((fit3[1] - fit3[0]) * 12 * 10),color='dimgrey', linewidth = 3,linestyle='--')

plt.gca().yaxis.set_label_position("right")
plt.gca().yaxis.tick_right()

plt.ylim(-1.0,1.0)
plt.yticks([-1,-0.8,-0.6,-0.4,-0.2,0,0.2,0.4,0.6,0.8,1])
plt.xlim(0,1.5)
plt.grid(zorder=10)

plt.gca().invert_xaxis()
plt.tight_layout()
f.savefig('core.dist.trend.pdf')

## Figure 8

In [ ]:
def detrend_linear(dat, dim, order):
    """ linear detrend dat along the axis dim """
    params = dat.polyfit(dim=dim, deg=order)
    fit = xr.polyval(dat[dim], params.polyfit_coefficients)
    dat = dat-fit
    return dat, fit

In [ ]:
data_use = core_cesm2.groupby('time.year').mean().isel(year=slice(0,150))

params = data_use.sel(year=slice('1950','2100')).polyfit(dim='year', deg=1)

polyval = xr.polyval(data_use, params.polyfit_coefficients)

In [ ]:
## Combined figure:

# compare Latitude of Core
f, ax = plt.subplots(3,1,figsize=(9,9), sharex=True)


###### top panel = latidue of core
dat, fit = detrend_linear(core_cesm2.mean('member'), 'time', 1)
fit.plot(color='white',linewidth=5,label='Ensemble Average', ax = ax[0], zorder=9)
fit.plot(color='k',linewidth=3,label='Ensemble Average', ax = ax[0], zorder=10)
(core_cesm2).plot.line(hue='member',color='lightblue',linewidth=0.1,alpha=0.9,zorder=0, ax = ax[0], add_legend=False)
(core_cesm2).mean('member').plot(color='cornflowerblue',linewidth=1,alpha=0.9,zorder=1, ax = ax[0], add_legend=False)

ax[0].set_xlim(core_cesm2.time[0],core_cesm2.time[-1])
ax[0].set_ylabel('latitude')
ax[0].set_xlabel('')
ax[0].set_ylim(36,47)
ax[0].grid()

###### bottom panel = latitude of bifu
dat, fit = detrend_linear(bifu_cesm2.mean('member'), 'time', 1)
fit.plot(color='white',linewidth=5,label='CESM2', ax = ax[1], zorder=9)
fit.plot(color='k',linewidth=3,label='CESM2', ax = ax[1], zorder=10)
(bifu_cesm2.lat).plot.line(hue='member',color='lightblue',linewidth=0.1,alpha=0.9,zorder=0, ax = ax[1], add_legend=False)
(bifu_cesm2.lat).mean('member').plot(color='cornflowerblue',linewidth=1,alpha=0.9,zorder=1, ax = ax[1], add_legend=False)
# (str_mer).plot(color='red',linewidth=1,label='SODA', ax = ax[2])

ax[1].set_ylim(37,53)
ax[1].set_title('')
ax[1].set_xlim(str_cesm2.time[0],str_cesm2.time[-1])

# ax[1].legend()
ax[1].set_ylabel('latitude')

ax[1].grid()
###### middle panel = strength (uvel)
dat, fit = detrend_linear((str_cesm2/100).mean('member'), 'time', 1)
fit.plot(color='white',linewidth=5,label='CESM2', ax = ax[2], zorder=9)
fit.plot(color='k',linewidth=3,label='CESM2', ax = ax[2], zorder=10)
(str_cesm2/100).plot.line(hue='member',color='lightblue',linewidth=0.1,alpha=0.9,zorder=0, ax = ax[2], add_legend=False)
(str_cesm2/100).mean('member').plot(color='cornflowerblue',linewidth=1,alpha=0.9,zorder=1, ax = ax[2], add_legend=False)
# (str_mer).plot(color='red',linewidth=1,label='SODA', ax = ax[2])
ax[1].set_xlabel('')

ax[2].set_title('')
ax[2].set_xlim(str_cesm2.time[0],str_cesm2.time[-1])

# ax[1].legend()
ax[2].set_ylabel('strength (m/s)')
ax[2].set_xlabel('')

ax[2].set_ylim(0.01,0.12)
ax[2].grid()
ax[0].set_xlim(core_cesm2.time[0],core_cesm2.time[-13])

plt.tight_layout()

f.savefig('indices.future.pdf')

In [ ]:
f, ax = plt.subplots(1,1,figsize=(6,5))

# use lines/steps
core_cesm2.sel(time=slice('1990','2000')).plot.hist(range=(35,50),bins=15,color='blue',alpha=0.5,histtype="step",linewidth=5,label = '1990s');
core_cesm2.sel(time=slice('2040','2050')).plot.hist(range=(35,50),bins=15,color='grey',alpha=0.5,histtype="step",linewidth=5, label = '2040s');
core_cesm2.sel(time=slice('2090','2100')).plot.hist(range=(35,50),bins=15,color='red',alpha=0.5,histtype="step",linewidth=5, label = '2090s');

plt.legend()
plt.title('')
plt.xlabel('latitude of core ($^o$N)')

f.savefig('core.hist.pdf')

In [ ]:
f, ax = plt.subplots(1,1,figsize=(6,5))

# use lines/steps
(str_cesm2/100).sel(time=slice('1990','2000')).plot.hist(range=(-2/100,15/100),bins=15,color='blue',alpha=0.5,histtype="step",linewidth=5,label = '1990s');
(str_cesm2/100).sel(time=slice('2040','2050')).plot.hist(range=(-2/100,15/100),bins=15,color='grey',alpha=0.5,histtype="step",linewidth=5, label = '2040s');
(str_cesm2/100).sel(time=slice('2090','2100')).plot.hist(range=(-2/100,15/100),bins=15,color='red',alpha=0.5,histtype="step",linewidth=5, label = '2090s');

plt.legend()
plt.xlabel('strength (m/s)')
plt.title('')

f.savefig('strength.hist.pdf')

In [ ]:
f, ax = plt.subplots(1,1,figsize=(6,5))

# use lines/steps
(bifu_cesm2.lat).sel(time=slice('1990','2000')).plot.hist(range=(30,55),bins=15,color='blue',alpha=0.5,histtype="step",linewidth=5,label = '1990s');
(bifu_cesm2.lat).sel(time=slice('2040','2050')).plot.hist(range=(30,55),bins=15,color='grey',alpha=0.5,histtype="step",linewidth=5, label = '2040s');
(bifu_cesm2.lat).sel(time=slice('2090','2100')).plot.hist(range=(30,55),bins=15,color='red',alpha=0.5,histtype="step",linewidth=5, label = '2090s');

plt.legend()
plt.xlabel('latitude ($^o$N)')
plt.title('')

f.savefig('bifu.hist.pdf')

## Figure 6-7

In [ ]:
## surfac
vmin_use = -0.8
vmax_use = 0.8
rolling_length='6'

lat_core6_surf = xr.open_dataset('lat_core.surface.6month.index.nc'); lat_core6_surf = lat_core6_surf.r.where(lat_core6_surf.p < 0.05)
lat_bifu6_surf = xr.open_dataset('lat_bifu.0m.6month.index.nc'); lat_bifu6_surf = lat_bifu6_surf.r.where(lat_bifu6_surf.p < 0.05)
str_core6_surf = xr.open_dataset('uvel.surface.6month.index.031126.nc'); str_core6_surf= str_core6_surf.r.where(str_core6_surf.p < 0.05)

lat_core6_250m = xr.open_dataset('lat_core.250m.' + rolling_length + 'month.index.nc'); lat_core6_250m = lat_core6_250m.r.where(lat_core6_250m.p < 0.05)
lat_bifu6_250m = xr.open_dataset('lat_bifu.250m.' + rolling_length + 'month.index.nc'); lat_bifu6_250m = lat_bifu6_250m.r.where(lat_bifu6_250m.p < 0.05)
str_core6_250m = xr.open_dataset('uvel.250m.' + rolling_length + 'month.index.nc'); str_core6_250m = str_core6_250m.r.where(str_core6_250m.p < 0.05)

r_calc_core_sig = r_calc_core.where(p_calc_core < 0.05)
r_calc_bifu_sig = r_calc_core.where(p_calc_bifu < 0.05)
r_calc_str_sig = r_calc_core.where(p_calc_str < 0.05)
r_calc_npgo_sig = r_calc_core.where(p_calc_npgo < 0.05)
r_calc_pdo_sig = r_calc_core.where(p_calc_pdo < 0.05)

In [ ]:
## SURFACE FOR ALL INDICES
f, ax = plt.subplots(3,5,figsize=(9,4),subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

lat_core6_surf.sel(data_var = 'pH').plot(ax = ax[0,0],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_core6_surf.sel(data_var = 'O2').plot(ax = ax[0,1],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_core6_surf.sel(data_var = 'NO3').plot(ax = ax[0,2],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_core6_surf.sel(data_var = 'TEMP').plot(ax = ax[0,3],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_core6_surf.sel(data_var = 'SSH').plot(ax = ax[0,4],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')

lat_bifu6_surf.sel(data_var = 'pH').plot(ax = ax[1,0],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_bifu6_surf.sel(data_var = 'O2').plot(ax = ax[1,1],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_bifu6_surf.sel(data_var = 'NO3').plot(ax = ax[1,2],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_bifu6_surf.sel(data_var = 'TEMP').plot(ax = ax[1,3],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_bifu6_surf.sel(data_var = 'SSH').plot(ax = ax[1,4],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')

str_core6_surf.sel(data_var = 'pH').plot(ax = ax[2,0],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
str_core6_surf.sel(data_var = 'O2').plot(ax = ax[2,1],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
str_core6_surf.sel(data_var = 'NO3').plot(ax = ax[2,2],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
str_core6_surf.sel(data_var = 'TEMP').plot(ax = ax[2,3],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
im = str_core6_surf.sel(data_var = 'SSH').plot(ax = ax[2,4],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')

ax = ax.flatten()
for i in range(len(ax)):
    gl = ax[i].gridlines(crs=ccrs.PlateCarree(), linewidth=1, linestyle='--', color='black', alpha=0.1); gl.top_labels = False; gl.right_labels = False
    ax[i].set_extent([180,270,10,70],crs = ccrs.PlateCarree())
    ax[i].add_feature(cfeature.LAND, color='k', zorder=3)
    ax[i].set_title('')    
    
plt.tight_layout()
#colorbar
f.subplots_adjust(right=0.83); cbar_ax = f.add_axes([0.85, 0.12, 0.035, 0.750]); cbar = f.colorbar(im, cax=cbar_ax, ticks=[-1,-0.5,0,0.5,1], label='significant correlation coefficient (r)'); cbar.ax.tick_params(labelsize=10)


f.savefig('SURF.core.BGC.pdf')

In [ ]:
## 250m FOR ALL INDICES
f, ax = plt.subplots(3,5,figsize=(9,4),subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

lat_core6_250m.sel(data_var = 'pH').plot(ax = ax[0,0],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_core6_250m.sel(data_var = 'O2').plot(ax = ax[0,1],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_core6_250m.sel(data_var = 'NO3').plot(ax = ax[0,2],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_core6_250m.sel(data_var = 'TEMP').plot(ax = ax[0,3],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_core6_250m.sel(data_var = 'SSH').plot(ax = ax[0,4],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')

lat_bifu6_250m.sel(data_var = 'pH').plot(ax = ax[1,0],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_bifu6_250m.sel(data_var = 'O2').plot(ax = ax[1,1],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_bifu6_250m.sel(data_var = 'NO3').plot(ax = ax[1,2],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_bifu6_250m.sel(data_var = 'TEMP').plot(ax = ax[1,3],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
lat_bifu6_250m.sel(data_var = 'SSH').plot(ax = ax[1,4],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')

str_core6_250m.sel(data_var = 'pH').plot(ax = ax[2,0],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
str_core6_250m.sel(data_var = 'O2').plot(ax = ax[2,1],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
str_core6_250m.sel(data_var = 'NO3').plot(ax = ax[2,2],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
str_core6_250m.sel(data_var = 'TEMP').plot(ax = ax[2,3],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')
im = str_core6_250m.sel(data_var = 'SSH').plot(ax = ax[2,4],vmin=vmin_use,vmax=vmax_use,cmap = 'coolwarm',add_colorbar=False,transform = ccrs.PlateCarree(),extend='both')

ax = ax.flatten()
for i in range(len(ax)):
    gl = ax[i].gridlines(crs=ccrs.PlateCarree(), linewidth=1, linestyle='--', color='black', alpha=0.1); gl.top_labels = False; gl.right_labels = False
    ax[i].set_extent([180,270,10,70],crs = ccrs.PlateCarree())
    ax[i].add_feature(cfeature.LAND, color='k', zorder=3)
    ax[i].set_title('')    

ax[4].remove()
ax[9].remove()
ax[14].remove()

plt.tight_layout()
#colorbar
f.subplots_adjust(right=0.83); cbar_ax = f.add_axes([0.85, 0.12, 0.035, 0.750]); cbar = f.colorbar(im, cax=cbar_ax, ticks=[-1,-0.5,0,0.5,1], label='significant correlation coefficient (r)'); cbar.ax.tick_params(labelsize=10)

f.savefig('250M.core.BGC.pdf')